In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType

catalogo = "will"
esquema = "i30"

tabela_bronze = f"{catalogo}.{esquema}.bronze_abastecimento"
tabela_silver = f"{catalogo}.{esquema}.silver_abastecimento"

# 1. Leitura dos dados da Bronze
df_bronze = spark.table(tabela_bronze)

# 2. Transformações e padronização para a Camada Silver
df_silver = (
    df_bronze
    .withColumn("DATA", F.to_date(F.col("DATA"), "dd/MM/yyyy"))
    # Mapeia e tipa os campos reais da Bronze de abastecimento
    .withColumn("POSTO_PRECO_LITRO", F.col("R$_LITRO").cast(DoubleType()))
    .withColumn("POSTO_VALOR_TOTAL", F.col("R$_TOTAL").cast(DoubleType()))
    # Enriquecimento: Se a coluna LITROS não existir ou vier vazia, calcula com base no valor total / preço litro
    .withColumn(
        "LITROS_ABASTECIDOS", 
        F.when(F.col("LITROS").isNotNull(), F.col("LITROS").cast(DoubleType()))
         .otherwise(F.round(F.col("POSTO_VALOR_TOTAL") / F.col("POSTO_PRECO_LITRO"), 2))
    )
    # Seleção final limpa
    .select(
        "DATA",
        "POSTO_PRECO_LITRO",
        "POSTO_VALOR_TOTAL",
        "LITROS_ABASTECIDOS",
        "_source_file"
    )
    .withColumn("_processed_at", F.current_timestamp())
)

# 3. Gravação na Camada Silver
(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(tabela_silver)
)

print(f"Camada Silver de abastecimento gravada com sucesso: {tabela_silver}")
display(spark.table(tabela_silver))